In [39]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [40]:
from pathlib import Path
import asyncio
from hedging_dataset_creator.hedging_labeller import hedging_labeller
from hedging_dataset_creator.sentence_tokenizer import sentence_tokenizer
from speech_parser.transcript_parser import parse_transcript_to_json
import pandas as pd

TRANSCRIPT_PATH = "data/Transcripts/AAPL/2016-Apr-26-AAPL.txt"
PROCESSED_JSON_PATH = Path("data/processed/2016-Apr-26-AAPL.json")

parse_transcript_to_json(TRANSCRIPT_PATH)
sentences = pd.Series(sentence_tokenizer(PROCESSED_JSON_PATH)['sentence'])
sentences

0                                             Thank you.
1      Good afternoon, and thanks to everyone for joi...
2      Speaking first is Apple's CEO, Tim Cook, and h...
3      After that, we'll open the call to questions f...
4      Please note that some of the information you'l...
                             ...                        
338    And the numbers for the telephone replay are 8...
339    These replays will be available by approximate...
340    Members of the press with additional questions...
341    Joan is at 408-974-4570, and I am at 408-974-5...
342                         Thanks again for joining us.
Name: sentence, Length: 343, dtype: object

In [41]:
from hedging_dataset_creator.hedging_labeller import get_hedging_on_sentence
print(get_hedging_on_sentence(list(sentences[:10])))

You are a financial language expert specialising in earnings call analysis.

    Label the following sentences as hedging (1) or not hedging (0).

    Hedging language includes any of the following:
    - Modal verbs expressing uncertainty: may, might, could, would, should
    - Epistemic verbs: believe, think, expect, anticipate, feel, assume
    - Approximators: approximately, around, roughly, about
    - Probability expressions: likely, unlikely, possibly, probably, perhaps
    - Conditional framing: assuming, subject to, if, provided that
    - Opportunity hedges: potential, opportunity, possibility
    - Attribution hedges: based on current trends, according to our estimates
    - Distancing language: it appears, it seems, there is reason to believe
    - Negative hedges: cannot guarantee, no assurance, cannot predict

    Important:
    - Not every modal verb is hedging. "We will pay the dividend"
    is NOT hedging. "We believe we will pay the dividend" IS hedging.
    Direct fa

In [44]:
print('Processing', len(sentences), 'sentences through Anthropic')
classified_sentences = await hedging_labeller(pd.DataFrame(sentences), semaphore=asyncio.Semaphore(10))
classified_sentences

Processing 343 sentences through Anthropic


Labelling hedging sentences: 100%|██████████| 35/35 [00:03<00:00,  8.89batch/s]


,sentence,isHedge
0,Thank you.,0
1,"Good afternoon, and thanks to everyone for joi...",0
2,"Speaking first is Apple's CEO, Tim Cook, and h...",0
3,"After that, we'll open the call to questions f...",0
4,Please note that some of the information you'l...,1
...,...,...
338,And the numbers for the telephone replay are 8...,0
339,These replays will be available by approximate...,1
340,Members of the press with additional questions...,0
341,"Joan is at 408-974-4570, and I am at 408-974-5...",0


In [45]:
classified_sentences.to_csv("data/hedging_dataset/AAPL/2016-Apr-26-AAPL.csv", index=False)